# **SMS Spam Detection**

student details:

Daniel Argunov 8909,
Yuval Vaknin 4475,
Liron Moussai 4233

---



# Part 1 – Introduction and Problem Description

## Problem Description

The goal of this project is to build a Machine Learning model that automatically classifies SMS text messages as either **Spam** or **Ham** (legitimate messages).

This is a **supervised binary classification** problem.

- **Input:** the text of an SMS message.
- **Target:** the message category – Spam or Ham.
- **Learning type:** Supervised Learning.
- **Task type:** Binary Classification.
- **Implemented algorithm:** Naive Bayes.

## Dataset

The dataset used in this project is the **Spam Text Message Classification** dataset from Kaggle.

Kaggle URL:  
https://www.kaggle.com/datasets/team-ai/spam-text-message-classification

The dataset contains SMS messages together with their corresponding labels:
- **ham** – legitimate message
- **spam** – spam message

The original Kaggle dataset is provided as a single labeled dataset. Therefore, after the initial data inspection and cleaning, it is split once into a training set and a held-out test set. The test set is not used during model training or hyperparameter selection.

# AI Assistance

ChatGPT was used as an assistance tool during the project.

## Purpose of Use

ChatGPT was used for:
- Understanding the assignment requirements.
- Checking whether the selected dataset meets the assignment requirements.
- Planning the Machine Learning workflow.
- Understanding the different stages of an NLP classification project.
- Explaining Machine Learning concepts and results.
- Reviewing the implementation against the assignment requirements.

## Prompts Used

The following prompts were used during the development of the project:

1. "Explain the Machine Learning assignment requirements and divide the project into tasks."
2. "Check whether the Spam Text Message Classification dataset from Kaggle is suitable for the assignment requirements."


## Dataset Loading and Initial Inspection

In this section, the dataset is loaded and inspected to understand its structure before starting the Machine Learning process.

In [ ]:
from google.colab import files
import pandas as pd
import io

uploaded = files.upload()

filename = next(iter(uploaded))

df = pd.read_csv(io.BytesIO(uploaded[filename]))

df.head()

Saving SPAM text message 20170820 - Data.csv to SPAM text message 20170820 - Data (1).csv


,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nClass distribution:")
print(df["Category"].value_counts())

print("\nClass distribution (%):")
print(df["Category"].value_counts(normalize=True) * 100)

Dataset shape: (5572, 2)

Columns: ['Category', 'Message']

Missing values:
Category    0
Message     0
dtype: int64

Duplicate rows:
415

Class distribution:
Category
ham     4825
spam     747
Name: count, dtype: int64

Class distribution (%):
Category
ham     86.593683
spam    13.406317
Name: proportion, dtype: float64


In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

print("Dataset shape after removing duplicates:", df.shape)

Dataset shape after removing duplicates: (5157, 2)


### Train-Test Split

The dataset is divided into a training set and a test set.

80% of the data is used for training and 20% is used for testing.

Stratified splitting is used in order to preserve the original proportion of Spam and Ham messages in both datasets.

In [ ]:
from sklearn.model_selection import train_test_split

trainset, testset = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["Category"]
)

trainset = trainset.reset_index(drop=True)
testset = testset.reset_index(drop=True)

print("Train set size:", trainset.shape)
print("Test set size:", testset.shape)

Train set size: (4125, 2)
Test set size: (1032, 2)


In [ ]:
print("First 5 rows of the Train Set:")
display(trainset.head())

print("First 5 rows of the Test Set:")
display(testset.head())

First 5 rows of the Train Set:


,Category,Message
0,ham,Saw Guys and Dolls last night with Patrick Swa...
1,ham,Nothing but we jus tot u would ask cos u ba gu...
2,ham,Oh really?? Did you make it on air? What's you...
3,spam,TheMob>Hit the link to get a premium Pink Pant...
4,spam,"New Mobiles from 2004, MUST GO! Txt: NOKIA to ..."


First 5 rows of the Test Set:


,Category,Message
0,ham,Can you plz tell me the ans. BSLVYL sent via f...
1,ham,You intrepid duo you! Have a great time and se...
2,ham,R u in this continent?
3,ham,"Cool, want me to go to kappa or should I meet ..."
4,ham,I'm meeting Darren...


# Evaluation Metric

The main evaluation metric used in this project is the **F1-score for the Spam class**.

The dataset is imbalanced, with significantly fewer Spam messages than Ham messages. Therefore, accuracy alone may provide a misleading evaluation of the model.

The F1-score combines:

- **Precision** – how many messages predicted as Spam are actually Spam.
- **Recall** – how many actual Spam messages were successfully detected.

Because Spam is the class of interest in this binary classification problem, the final model is evaluated primarily using the **F1-score with Spam as the positive class**.

Precision and Recall are also reported to provide additional information about the model's performance.

# Part 2 – Feature Engineering

The textual SMS messages must be converted into numerical features before they can be processed by the learning algorithm.

In this project, **Bag of Words (BoW)** is used through `CountVectorizer`.

Each feature represents a word from the training vocabulary, and the value of the feature is the number of times that word appears in a message.

The vocabulary is learned **only from the training data**. The same learned vocabulary is then used to transform unseen data.

This approach is suitable for Multinomial Naive Bayes because the model works naturally with word-frequency/count features.

In [ ]:
X_train = trainset["Message"]
y_train = trainset["Category"]

X_test = testset["Message"]
y_test = testset["Category"]

print("Training examples:", len(X_train))
print("Test examples:", len(X_test))

Training examples: 4125
Test examples: 1032


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)
feature_names = vectorizer.get_feature_names_out()
print("Train Bag of Words shape:", X_train_bow.shape)
print("Test Bag of Words shape:", X_test_bow.shape)
print("Vocabulary size:", len(feature_names))

Train Bag of Words shape: (4125, 7674)
Test Bag of Words shape: (1032, 7674)
Vocabulary size: 7674


In [ ]:
def show_bow_examples(texts, labels, vectorizer, n_examples=3):
    """
    Display Bag-of-Words representations for several text examples.
    Only non-zero word counts are shown to keep the output readable.
    """

    texts = texts.reset_index(drop=True)
    labels = labels.reset_index(drop=True)

    transformed = vectorizer.transform(texts.iloc[:n_examples])
    feature_names = vectorizer.get_feature_names_out()

    examples = []

    for i in range(n_examples):
        row = transformed.getrow(i)

        bow_representation = {
            feature_names[index]: int(count)
            for index, count in zip(row.indices, row.data)
        }

        examples.append({
            "Message": texts.iloc[i],
            "Category": labels.iloc[i],
            "Bag of Words": bow_representation
        })

    return pd.DataFrame(examples)

In [ ]:
print("Feature Engineering examples from the TRAIN set:")
display(
    show_bow_examples(
        X_train,
        y_train,
        demo_vectorizer,
        n_examples=3
    )
)

print("\nFeature Engineering examples from the TEST set:")
display(
    show_bow_examples(
        X_test,
        y_test,
        demo_vectorizer,
        n_examples=3
    )
)

Feature Engineering examples from the TRAIN set:


,Message,Category,Bag of Words
0,Saw Guys and Dolls last night with Patrick Swa...,ham,"{'and': 1, 'dolls': 1, 'great': 1, 'guys': 1, ..."
1,Nothing but we jus tot u would ask cos u ba gu...,ham,"{'already': 1, 'ask': 1, 'ba': 1, 'but': 2, 'c..."
2,Oh really?? Did you make it on air? What's you...,ham,"{'air': 1, 'did': 1, 'it': 1, 'make': 1, 'oh':..."



Feature Engineering examples from the TEST set:


,Message,Category,Bag of Words
0,Can you plz tell me the ans. BSLVYL sent via f...,ham,"{'ans': 1, 'bslvyl': 1, 'can': 1, 'com': 1, 'f..."
1,You intrepid duo you! Have a great time and se...,ham,"{'and': 1, 'both': 1, 'great': 1, 'have': 1, '..."
2,R u in this continent?,ham,"{'in': 1, 'this': 1}"


# Part 3 – Multinomial Naive Bayes Implementation

Multinomial Naive Bayes is implemented from scratch as the learning algorithm for this project.

The algorithm estimates:

1. The prior probability of each class, \(P(C)\).
2. The probability of each word given a class, \(P(word|C)\).

For a new SMS message, the model combines the class prior with the probabilities of the words appearing in the message and predicts the class with the highest score.

**Laplace smoothing** is used through the `alpha` hyperparameter to avoid zero probabilities for words that were not observed in a specific class.

Log probabilities are used to avoid numerical underflow when combining many small probabilities.

The model provides separate `fit` and `predict` operations as required.

In [ ]:
import numpy as np


class NaiveBayesClassifier:

    def __init__(self, alpha=1.0):
        self.alpha = alpha


    def fit(self, X, y):

        y = np.asarray(y)

        self.classes_, class_counts = np.unique(
            y,
            return_counts=True
        )

        n_samples, n_features = X.shape

        self.class_log_prior_ = np.log(
            class_counts / n_samples
        )

        self.feature_log_prob_ = np.zeros(
            (len(self.classes_), n_features)
        )

        for i, current_class in enumerate(self.classes_):

            X_class = X[y == current_class]

            word_counts = np.asarray(
                X_class.sum(axis=0)
            ).ravel()

            total_words = word_counts.sum()

            probabilities = (
                word_counts + self.alpha
            ) / (
                total_words + self.alpha * n_features
            )

            self.feature_log_prob_[i] = np.log(
                probabilities
            )

        return self

    def predict(self, X):

        scores = (
            X @ self.feature_log_prob_.T
            + self.class_log_prior_
        )
        predicted_class_indices = np.asarray(
            scores.argmax(axis=1)
        ).ravel()

        return self.classes_[predicted_class_indices]

In [ ]:
def train_model(X_train, y_train, alpha=1.0):

    model = NaiveBayesClassifier(alpha=alpha)

    model.fit(X_train, y_train)

    return model


def predict_model(model, X):

    return model.predict(X)

## Part 4 - Training and Hyperparameter Selection

In this stage, the Naive Bayes model is trained using different values of the
`alpha` hyperparameter.

The `alpha` parameter controls the amount of smoothing applied to the word
probabilities.

Several alpha values are evaluated in order to select the value that provides
the best classification performance.

The test set is not used during hyperparameter selection. It is kept separately
for the final evaluation of the selected model.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

alpha_values = [0.1, 0.5, 1.0, 2.0]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for alpha in alpha_values:

    fold_scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        fold_vectorizer = CountVectorizer()

        X_fold_train_bow = fold_vectorizer.fit_transform(X_fold_train)
        X_fold_val_bow = fold_vectorizer.transform(X_fold_val)

        model = NaiveBayesClassifier(alpha=alpha)

        model.fit(
            X_fold_train_bow,
            y_fold_train
        )

        predictions = model.predict(
            X_fold_val_bow
        )

        score = f1_score(
            y_fold_val,
            predictions,
            pos_label="spam"
        )

        fold_scores.append(score)

    mean_f1 = np.mean(fold_scores)

    results.append({
        "Alpha": alpha,
        "Mean F1 Score": mean_f1
    })

results_df = pd.DataFrame(results)

results_df

,Alpha,Mean F1 Score
0,0.1,0.945541
1,0.5,0.941421
2,1.0,0.937452
3,2.0,0.919826


In [ ]:
best_row = results_df.loc[
    results_df["Mean F1 Score"].idxmax()
]

best_alpha = best_row["Alpha"]
best_f1 = best_row["Mean F1 Score"]

print("Best alpha:", best_alpha)
print("Best validation F1:", best_f1)

Best alpha: 0.1
Best validation F1: 0.9455407586814202


## Final Model Training

After selecting the best `alpha` value using 5-fold cross-validation, the final Naive Bayes model is trained on the complete training set.

The Bag-of-Words representation used for the final model was learned from the complete training set only. The test set was transformed using the same vocabulary and was not used during model training or hyperparameter selection.

In [ ]:
final_model = NaiveBayesClassifier(
    alpha=best_alpha
)

final_model.fit(
    X_train_bow,
    y_train
)

print("Final model trained with alpha =", best_alpha)

Final model trained with alpha = 0.1


# Part 5 – Final Evaluation on the Test Set

After selecting the best `alpha` value and training the final model on the complete training set, the model is evaluated on the held-out test set.

The test set was not used during model training or hyperparameter selection.

The main evaluation metric is the **F1-score for the Spam class**.

In [ ]:
y_test_pred = final_model.predict(X_test_bow)

print("Prediction completed.")

Prediction completed.


In [ ]:
prediction_results = pd.DataFrame({
    "Message": X_test.reset_index(drop=True),
    "Actual": y_test.reset_index(drop=True),
    "Predicted": y_test_pred
})

prediction_results.head(5)

,Message,Actual,Predicted
0,Can you plz tell me the ans. BSLVYL sent via f...,ham,ham
1,You intrepid duo you! Have a great time and se...,ham,ham
2,R u in this continent?,ham,ham
3,"Cool, want me to go to kappa or should I meet ...",ham,ham
4,I'm meeting Darren...,ham,ham


In [ ]:
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)

test_precision = precision_score(
    y_test,
    y_test_pred,
    pos_label="spam"
)

test_recall = recall_score(
    y_test,
    y_test_pred,
    pos_label="spam"
)

test_f1 = f1_score(
    y_test,
    y_test_pred,
    pos_label="spam"
)

final_metrics = pd.DataFrame({
    "Metric": [
        "Precision - Spam",
        "Recall - Spam",
        "F1 Score - Spam"
    ],
    "Score": [
        test_precision,
        test_recall,
        test_f1
    ]
})

final_metrics["Score"] = final_metrics["Score"].round(4)

final_metrics

,Metric,Score
0,Precision - Spam,0.9355
1,Recall - Spam,0.9062
2,F1 Score - Spam,0.9206


## Evaluation Summary

The final model is evaluated using the F1-score for the Spam class, together with Precision and Recall.

- **Precision** measures how reliable the model's Spam predictions are.
- **Recall** measures how many of the actual Spam messages are detected.
- **F1-score** provides a balance between Precision and Recall.

The test set was used only after the model structure and the `alpha` hyperparameter had already been selected using the training set.

Therefore, the reported test performance represents the model's ability to generalize to previously unseen SMS messages.